## Scryfall Bulk Data Curation Pipeline

This notebook can be used to help gather, filter, and clean *Magic the Gathering* card data from Scryfall.

In [ ]:
import os
import sys

RAW_PATH = './data/default-cards.json'
PROCESSED_PATH = './data/cards_full.ndjson'

def ensure_dataset():
    os.makedirs('./data', exist_ok=True)

    # Download the dataset if it is missing
    if not os.path.exists(RAW_PATH):
        print('Raw dataset not found. Downloading...')

        IN_COLAB = 'google.colab' in sys.modules

        # If using Colab make an API call and write the file
        if IN_COLAB:
            import requests

            res = requests.get('https://api.scryfall.com/bulk-data').json()
            download_url = next(
                item['download_uri']
                for item in res['data']
                if item['type'] == 'default_cards'
            )

            r = requests.get(download_url)
            with open(RAW_PATH, 'wb') as f:
                f.write(r.content)

        # Otherwise run the download script    
        else:
            sys.path.append('./src')
            from download_data import download_data
            download_data()

    else:
        print('Raw dataset already exists.')

    # Process the dataset
    if not os.path.exists(PROCESSED_PATH):
        print('Processed dataset not found. Creating creatures dataset...')

        import json
        
        try:
            import ijson
        except ImportError:
            print('ijson not found. Installing...')

            import subprocess
            subprocess.check_call([sys.executable, 'm', 'pip', 'install', 'ijson'])

            import ijson

        # Remove certain cards not relevant to analysis, like art, tokens, joke sets, and malformed entries    
        with open(RAW_PATH, 'rb') as f, open(PROCESSED_PATH, 'w') as out:
            for card in ijson.items(f, 'item'):
                if not card.get('type_line'):
                    continue
                    
                if card.get('lang') != 'en':
                    continue
                    
                if 'token' in card.get('layout'):    
                    continue
                    
                if card.get('digital', False):
                    continue

                if card.get('set_type') in ['funny']:
                    continue

                if card.get('set_type') in ['memorabilia']:
                    continue
                
                out.write(json.dumps(card) + '\n')
        
        print('Processing complete!')
    
    else:
        print('Processed dataset already exists.')

In [ ]:
import pandas as pd
import numpy as np

cards_df = pd.read_json(PROCESSED_PATH, lines=True)

This section adds some columns that help us with preprocessing the dataset for filtering. You can expand on these filters if you are especially interested in certain features.

In [ ]:
# Optimize the dataset for filtering
cards_df['type_line_lower'] = cards_df['type_line'].str.lower()

cards_df['power_num'] = pd.to_numeric(cards_df['power'], errors='coerce')
cards_df['toughness_num'] = pd.to_numeric(cards_df['toughness'], errors='coerce')

cards_df['oracle_lower'] = cards_df['oracle_text'].str.lower()

cards_df['keywords_joined'] = cards_df['keywords'].apply(
    lambda x: ' '.join(x).lower() if isinstance(x, list) else ''
)

We have a function to translate Scryfall-like user queries into filters in pandas. Additional filters may be added at a user's discretion.

In [ ]:
import re

# function to allow for queries similar to Scryfall
def fast_query(df, query):
    tokens = query.split()
    mask = pd.Series(True, index=df.index)

    order_field = None
    ascending = True

    for token in tokens:

        # filter by typeline
        if token.startswith('type:'):
            value = token.split(':')[1]
            mask &= df['type_line_lower'].str.contains(value)
            
        # filter by set
        elif token.startswith('set:'):
            value = token.split(':')[1]
            mask &= df['set'] == value

        # filter by keyword
        elif token.startswith('keyword:'):
            value = token.split(':')[1]
            mask &= df['keywords_joined'].str.contains(value, na=False)

        # filter by oracle text
        elif token.startswith('oracle:'):
            value = token.split(':')[1]
            mask &= df['oracle_lower'].str.contains(value)

        # filter by language
        elif token.startswith('lang:'):
            value = token.split(':')[1]
            mask &= df['lang'].str.contains(value)

        # order query under different conditions
        elif token.startswith('order:'):
            field = token.split(':')[1]

            if field.startswith('-'):
                ascending = False
                order_field = field[1:]
            else:
                order_field = field

        # filter by mana value
        elif re.match(r'cmc[<>=]+', token):
            op = re.findall(r'[<>=]+', token)[0]
            value = float(token.split(op)[1])

            if op == '<=':
                mask &= df['cmc'] <= value
            elif op == '<':
                mask &= df['cmc'] < value
            elif op == '>=':
                mask &= df['cmc'] >= value
            elif op == '>':
                mask &= df['cmc'] > value
            elif op == '=':
                mask &= df['cmc'] == value

    result = df[mask]
    result = result.drop_duplicates(subset='oracle_id')

    if order_field:
        result = result.sort_values(order_field, ascending=ascending)

    return result

In [ ]:
# Columns (features) we are interested in
base_cols = [
    'name',
    'cmc',
    'oracle_text',
    'keywords',
    'set',
    'set_name',
    'type_line'
]

creature_cols = ['power', 'toughness']

# Structured query to grab all creature cards in the Khans of Tarkir set, ordered by mana value (cmc)
query = 'set:ktk type:creature order:cmc'

result = fast_query(cards_df, query)

cols = [c for c in base_cols if c in result.columns]

if result['type_line'].str.contains('Creature', na=False).any():
    cols += [c for c in creature_cols if c in result.columns]
    
result = result[cols].copy()

# Name this however you want
filename = 'Creatures.csv'
#result.to_csv(filename, index=False) <- uncomment this out to write the files to your directory
#print(f'Saved {filename}')